# Speed Dating data
Can't know how to clean it


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_rows = 1000
pd.options.display.max_columns = 200
pd.set_option('display.max_columns', None)

In [ ]:
#%pip install ydata-profiling

In [ ]:
# Load data
data = pd.read_csv('Speed_Dating_Data.csv', sep=',', encoding='ISO-8859-1')
data.head(20)

In [ ]:
data.isnull().sum()

In [ ]:
data.isnull().mean() * 100

In [ ]:
# 1. BẢNG THÔNG TIN CÁ NHÂN (PERSON_INFO)
# Chứa các đặc điểm nhân khẩu học, học vấn và lối sống (Cố định cho mỗi iid)
# =================================================================
personal_cols = [
    'iid', 'gender', # Định danh và nhóm thử nghiệm
    'age', 'field', 'field_cd',          # Học vấn: Tuổi, ngành học, trường ĐH
    # 'mn_sat', 'tuition', 'undergra',
    'race', 'imprace',          # Kinh tế/Sắc tộc: Điểm SAT, học phí, tầm quan trọng sắc tộc
    'imprelig', 
    #'from',         # Tôn giáo, quê quán,
     # 'zipcode', 'income',       #thu nhập gia đình
    'goal', 'date', 'go_out', 'career', 'career_c', # Mục tiêu tham gia, tần suất hẹn hò, nghề nghiệp
    'sports', 'tvsports', 'exercise', 'dining',     # Sở thích (1-10): Thể thao, ăn uống...
    'museums', 'art', 'hiking', 'gaming', 'clubbing', 
    'reading', 'tv', 'theater', 'movies', 'concerts', 
    'music', 'shopping', 'yoga',
    # 'exphappy', # Kỳ vọng mức độ hạnh phúc (1-10)
    # 'expnum',   # Dự đoán số người sẽ thích mình (trong 20 người)
    # 'match_es', # Dự đoán số lượng match sẽ có
    'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1', # Tìm kiếm ở đối phương
    'attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1', 'shar2_1',         # Nghĩ giới kia tìm gì
    'attr3_1', 'sinc3_1', 'intel3_1', 'fun3_1', 'amb3_1',                   # Tự đánh giá mình
]



In [ ]:
interaction_cols = [
    'iid', 'pid', 'int_corr', 'samerace',
    # Thông tin đối tác (hậu tố _o)
    # Đánh giá của đối tác về người đó (hậu tố _o)
    'dec_o', 'attr_o', 'sinc_o', 'intel_o', 'fun_o', 'amb_o', 'shar_o', 'like_o', 'prob_o',
    # Đánh giá của chính người đó về đối tác (Scorecard)
    'dec', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar', 'like', 'prob',
    'match' # Kết quả liên lạc thực tế
] 

In [ ]:
data[interaction_cols].drop_duplicates().head(20)

In [ ]:
# Tập hợp tất cả các cột đã phân loại
classified_cols = set(personal_cols + interaction_cols)

# Các cột còn lại chưa được đưa vào bảng nào
remaining_cols = [col for col in data.columns if col not in classified_cols]

print(f"Các cột chưa phân loại: {remaining_cols}")

In [ ]:
df = data.drop(columns=[col for col in data.columns if col not in classified_cols]).copy()
df = df.drop(df.index[df['pid'].isnull()])  # Loại bỏ các hàng có pid bị thiếu
df['income'] = pd.to_numeric(df['income'].str.replace(',', ''), errors='coerce')


In [ ]:
df_users = df[personal_cols].drop_duplicates(subset='iid').reset_index(drop=True).copy()
df_users.info()
df_users.head(20)

In [ ]:
df_interactions = df[interaction_cols].copy()
df_interactions['pid'] = df_interactions['pid'].astype(int)
df_interactions.info()
df_interactions.head(20)

In [ ]:
# def calculate_int_corr(df, iid, pid):
#     # 1. Danh sách 17 cột sở thích (Activities)
#     activity_cols = [
#         'sports', 'tvsports', 'exercise', 'dining', 'museums', 'art', 
#         'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater', 
#         'movies', 'concerts', 'music', 'shopping', 'yoga',
#     ]
    
#     # 2. Lấy vector sở thích của User (iid)
#     user_profile = df[df['iid'] == iid][activity_cols].iloc[0]
    
#     # 3. Lấy vector sở thích của Partner (pid)
#     # Lưu ý: Trong dataset, pid cũng là một iid của người khác
#     partner_profile = df[df['iid'] == pid][activity_cols].iloc[0]
    
#     # 4. Tính toán hệ số tương quan Pearson
#     # Chúng ta dùng hàm .corr() của Series hoặc np.corrcoef
#     correlation = user_profile.corr(partner_profile)
    
#     return correlation
# calculate_int_corr(df_users, 2, 12)

In [ ]:
df_users[df_users['sports'].isnull()].head(30)

In [ ]:
df_users.dropna(subset=['sports'], inplace=True)
df_users['imprelig'].head(30)


In [ ]:
df_users.isnull().sum()

In [ ]:
check = df_users.select_dtypes(include=['number'])
print(f"Dropped columns: {df_users.columns.difference(check.columns).tolist()}")

In [ ]:
# Drop non-numeric columns
data_numeric = data.select_dtypes(include=['object'])
print(data_numeric.info())
print(data_numeric.isnull().mean())
# data_numeric.head(50)
# in tỷ lệ null




# EDA auto


In [ ]:
# from ydata_profiling import ProfileReport

# # Pass the DataFrame you want to perform EDA on to the ProfileReport function of Ydata-Profiling.
# profile_df = ProfileReport(df_users,title="Personal Data Report", explorative=True)

# # Executing this will automatically save a visualization report file in the current directory.
# profile_df.to_notebook_iframe()